# Arabic Contextual Word Embeddings: Do Models Understand Word Sense?

Arabic has many words whose meaning depends entirely on context. **هدف** can be a *goal* scored in a football match or a company's *objective*. **شبكة** can be the *net* of the goal or an internet *network*.

Static embeddings (Word2Vec, GloVe) give each word a single vector no matter where it appears. Transformer models produce **contextual** embeddings, where the vector changes with the surrounding words. This project measures how well that works for Arabic by comparing four model families:

| Model | Type | Notes |
|---|---|---|
| [AraBERT v02](https://huggingface.co/aubmindlab/bert-base-arabertv02) | Encoder (BERT) | Trained specifically on Arabic |
| [BGE-M3](https://huggingface.co/BAAI/bge-m3) | Encoder (XLM-RoBERTa) | Multilingual retrieval model, 8K context |
| [multilingual-E5-base](https://huggingface.co/intfloat/multilingual-e5-base) | Encoder (XLM-RoBERTa) | Multilingual embedding model, expects a `passage: ` prefix |
| [Qwen2.5](https://huggingface.co/Qwen/Qwen2.5-1.5B) | Decoder (causal LLM) | General-purpose LLM, byte-level BPE tokenizer |

**What the notebook covers**
1. How each tokenizer splits Arabic text
2. Extracting token and word embeddings from the last hidden layer
3. A word-sense test: does the same word get closer vectors when the meaning is the same?
4. Frozen vs. trainable embedding layers
5. What happens when the input exceeds the model's maximum length

## 1. Setup

In [ ]:
import json
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers.utils import logging as hf_logging

from src.contextual_embeddings import (
    MODELS, cosine, evaluate_polysemy, get_device, load_model,
    free_memory, sentence_embedding, summarize, tokenization_report, word_embedding,
)

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()
pd.set_option("display.max_colwidth", 120)

# Qwen2.5-7B needs a GPU with ~16 GB of memory (e.g. Colab T4 in bfloat16).
# Leave this off to run everything else on a normal laptop.
RUN_LARGE_MODELS = False

device = get_device()
print("Device:", device)

## 2. Data

The two sentences below are the core example. Each contains both **هدف** and **شبكة**, but with different meanings:

- *سجل اللاعب هدف الفوز بعد أن هزت الكرة شبكة المرمى*: the player scored the winning **goal** after the ball shook the **net**.
- *هدف الشركة بناء شبكة إنترنت أسرع في المدن*: the company's **objective** is to build a faster internet **network** in cities.

For a fair test, every polysemous word also has a third sentence that uses it in the *same* sense as the first one. That lets us check whether same-sense pairs end up closer than different-sense pairs.

In [ ]:
news = [
    "سجل اللاعب هدف الفوز بعد أن هزت الكرة شبكة المرمى",
    "هدف الشركة بناء شبكة إنترنت أسرع في المدن",
]

with open("data/polysemous_words.json", encoding="utf-8") as f:
    polysemous_words = json.load(f)

pd.DataFrame(polysemous_words)[["word", "sense_a", "sense_b", "anchor", "same_sense", "different_sense"]]

## 3. A closer look with AraBERT

Before comparing models, we walk through the pipeline step by step with AraBERT: tokenize, run the model, and read the token vectors from the last hidden state.

In [ ]:
arabert = MODELS[0]
tokenizer, model = load_model(arabert, device)

inputs = tokenizer(news, padding=True, return_tensors="pt").to(device)
print("Max sequence length:", tokenizer.model_max_length)

for i, ids in enumerate(inputs["input_ids"]):
    print(f"Sentence {i + 1}:", tokenizer.convert_ids_to_tokens(ids))

In [ ]:
with torch.no_grad():
    last_hidden_state = model(**inputs).last_hidden_state

# (batch_size, sequence_length, hidden_size)
print("Last hidden state shape:", tuple(last_hidden_state.shape))

# First 5 values of each token vector in the first sentence
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
for token, vector in zip(tokens, last_hidden_state[0]):
    print(f"{token:>10}  {vector[:5].cpu().numpy().round(3)}")

### Same word, different vectors

With a static embedding, **هدف** would have one fixed vector. Here we pull its contextual vector from each sentence and compare them. A similarity well below 1.0 means the model represents the two uses differently.

In [ ]:
for word in ["هدف", "شبكة"]:
    v1 = word_embedding(news[0], word, tokenizer, model)
    v2 = word_embedding(news[1], word, tokenizer, model)
    print(f"{word}: cosine similarity between the two contexts = {cosine(v1, v2):.4f}")

sentence_vectors = sentence_embedding(news, tokenizer, model)
print(f"\nSentence-level similarity = {cosine(sentence_vectors[0], sentence_vectors[1]):.4f}")

## 4. Frozen vs. trainable embeddings

When fine-tuning, a common choice is to freeze the input embedding layer so only the upper layers adapt. This saves memory and helps prevent overfitting on small datasets. In PyTorch this is just a matter of switching `requires_grad` off for that layer's parameters.

In [ ]:
embedding_layer = model.get_input_embeddings()
count = lambda: sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Embedding matrix shape:", tuple(embedding_layer.weight.shape))
print(f"Trainable parameters (all):              {count():,}")

for p in embedding_layer.parameters():
    p.requires_grad = False
print(f"Trainable parameters (embeddings frozen): {count():,}")

for p in embedding_layer.parameters():
    p.requires_grad = True
print(f"Trainable parameters (unfrozen again):    {count():,}")

## 5. Inputs longer than the model's limit

BERT-style models have a fixed number of position embeddings (512 for AraBERT). Feeding a longer sequence fails, so long documents must be truncated or split into chunks.

In [ ]:
long_text = " جامعة الحسين التقنية من أفضل الجامعات في الأردن" * 100

long_inputs = tokenizer(long_text, return_tensors="pt").to(device)
print("Tokens:", long_inputs["input_ids"].shape[1], "| model limit:", tokenizer.model_max_length)

# On CPU this raises a normal Python error. On a GPU, the same out-of-range index triggers a
# CUDA device-side assert that breaks the session, so we only try it on CPU.
if device.type == "cpu":
    try:
        with torch.no_grad():
            model(**long_inputs)
    except Exception as e:
        print(f"Without truncation -> {type(e).__name__}: position index out of range")

long_inputs = tokenizer(long_text, truncation=True, max_length=512, return_tensors="pt").to(device)
with torch.no_grad():
    print("With truncation    ->", tuple(model(**long_inputs).last_hidden_state.shape))

del model, tokenizer
free_memory()

## 6. Comparing the four models

For each model we run two experiments:

1. **Tokenization**: how many tokens each sentence turns into. Arabic-specific vocabularies usually keep whole words, while byte-level tokenizers (Qwen) break Arabic into many small pieces.
2. **Word-sense test**: for each polysemous word we compute
   - `sim_same_sense`: similarity between the word in two sentences with the **same** meaning
   - `sim_different_sense`: similarity between the word in two sentences with **different** meanings
   - `sense_separation = sim_same_sense - sim_different_sense`

A positive separation means the model places same-sense uses closer together, which is what we want. `accuracy` is the share of words where this holds.

> One thing to watch: Qwen is a **causal** model, so each token only sees the words *before* it. When the ambiguous word is the first word of the sentence (like **هدف** in the second sentence), Qwen has no context at all to disambiguate it.

In [ ]:
all_results, all_tokenization = [], []

for spec in MODELS:
    if spec.large and not RUN_LARGE_MODELS:
        print(f"Skipping {spec.label} (set RUN_LARGE_MODELS = True to include it)")
        continue

    print(f"Running {spec.label} ...")
    tokenizer, model = load_model(spec, device)

    tok = tokenization_report(news, tokenizer, spec.prefix)
    tok.insert(0, "model", spec.label)
    all_tokenization.append(tok)

    all_results.append(evaluate_polysemy(spec, tokenizer, model, polysemous_words))
    del model, tokenizer
    free_memory()

tokenization = pd.concat(all_tokenization, ignore_index=True)
results = pd.concat(all_results, ignore_index=True)

### Tokenization

In [ ]:
tokenization[["model", "sentence", "words", "tokens", "tokens_per_word"]]

In [ ]:
# Example of what the tokens actually look like for the first sentence
tokenization[tokenization.sentence == news[0]][["model", "token_list"]]

### Word-sense results

In [ ]:
results

In [ ]:
summary = summarize(results)
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pivot = results.pivot(index="word", columns="model", values="sense_separation")
pivot = pivot[summary.index]  # keep the same model order as the summary
pivot.plot(kind="bar", ax=axes[0], rot=0)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("Sense separation per word (higher is better)")
axes[0].set_ylabel("sim(same sense) - sim(different sense)")
axes[0].set_xlabel("")

summary["avg_sense_separation"].plot(kind="barh", ax=axes[1], color="tab:blue")
axes[1].invert_yaxis()
axes[1].set_title("Average sense separation by model")
axes[1].set_xlabel("sense separation")

plt.tight_layout()
plt.savefig("results/sense_separation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
results.to_csv("results/word_sense_results.csv", index=False, encoding="utf-8-sig")
summary.to_csv("results/model_summary.csv", encoding="utf-8-sig")
tokenization.drop(columns="token_list").to_csv("results/tokenization.csv", index=False, encoding="utf-8-sig")
print("Saved results to results/")

## 7. Takeaways

- **Contextual embeddings are not fixed per word.** The same Arabic word gets different vectors in different sentences, which static embeddings cannot do.
- **Raw cosine values are not comparable across models.** Each model has its own embedding space (for example, E5 vectors tend to all be quite similar to each other). That is why this project compares *separation* (the gap between same-sense and different-sense similarity) rather than raw similarity.
- **Tokenization matters for Arabic.** A model with an Arabic-aware vocabulary keeps most words whole, while byte-level BPE splits them into many fragments. This raises sequence length and cost.
- **Encoders vs. decoders.** Bidirectional encoders see the whole sentence, while a causal LLM only sees the left context. That is a real limitation when the ambiguous word comes early in the sentence.
- **Practical limits.** Inputs above the model's maximum length have to be truncated or chunked, and freezing the embedding layer is a cheap way to reduce trainable parameters during fine-tuning.

**Possible extensions:** a larger word-sense dataset, comparing layers other than the last one, visualizing embeddings with PCA/t-SNE, and adding static baselines such as AraVec or fastText.